# Assignment 1 2AMM10 2025-2026

## Group: Transistor
### Member 1: Clara Montemurro
### Member 2: Mirco Terenzi
### Member 3: Tommaso Zanotti

In [ ]:
import sys

if "google.colab" in sys.modules:
    pass
    !pip -q install pytorch-lightning pytorch-metric-learning

In [ ]:
import os
import re
from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output, display
from PIL import Image
from pytorch_metric_learning import losses, miners, samplers
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

DATASET_PATH = kagglehub.dataset_download("moltean/fruits")

pl.seed_everything(6)

FAST_DEV_RUN=False

training_transform = transforms.Compose(
    [
        transforms.Resize(size=(96, 96)),
        transforms.ToTensor(),
    ]
)

## Task 1
### Dataset

In [ ]:
class AppleDataset(Dataset):
    def __init__(self, transform=None, subset="train", class_subset="main"):
        assert subset in ["train", "test"]
        assert class_subset in ["main", "new", "all"]
        base = (
            Path(DATASET_PATH) / "fruits-360_original-size" / "fruits-360-original-size"
        )
        if subset == "train":
            self.path = base / "Training"
        elif subset == "test":
            self.path = base / "Validation"
        self.transform = transform
        all_folders = sorted(os.listdir(self.path))
        self.item_folders = sorted(
            x for x in all_folders if x.lower().startswith("apple")
        )
        generator = np.random.default_rng(6)
        generator.shuffle(self.item_folders)
        if class_subset == "main":
            self.item_folders = self.item_folders[:20]
        elif class_subset == "new":
            self.item_folders = self.item_folders[20:]
        self.targets = []
        self.image_paths = []
        for i, folder in enumerate(self.item_folders):
            for img_file in sorted(os.listdir(self.path / folder)):
                if img_file.startswith("r0"):
                    if class_subset == "new":
                        self.targets.append(i + 20)
                    else:
                        self.targets.append(i)
                    self.image_paths.append(self.path / folder / img_file)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        image = Image.open(self.image_paths[i]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.targets[i]

In [ ]:
%%script echo skipping dataset visualization
dataset = AppleDataset()

item_dd = widgets.Dropdown(options=dataset.item_folders, description="Variety:")
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame:")
output = widgets.Output()


def get_frames(folder):
    return sorted(
        f
        for f in os.listdir(dataset.path / folder)
        if f.startswith("r0_") and f.endswith(".jpg")
    )


def update_slider(*_):
    frames = get_frames(item_dd.value)
    frame_slider.max = max(0, len(frames) - 1)
    frame_slider.value = min(frame_slider.value, frame_slider.max)
    show_image()


def show_image(*_):
    frames = get_frames(item_dd.value)
    if not frames or frame_slider.value >= len(frames):
        return
    with output:
        clear_output(wait=True)
        img = Image.open(dataset.path / item_dd.value / frames[frame_slider.value])
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(img)
        ax.set_title(f"{item_dd.value} | frame {frame_slider.value}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()


item_dd.observe(update_slider, names="value")
frame_slider.observe(show_image, names="value")

update_slider()
display(widgets.VBox([item_dd, frame_slider, output]))

In [ ]:
train_data = AppleDataset(subset="train", transform=training_transform)
test_data = AppleDataset(subset="test", transform=training_transform)
support_new_data = AppleDataset(subset="train", transform=training_transform, class_subset="new")
test_new_data = AppleDataset(subset="test", transform=training_transform, class_subset="new")

### Implementation

In [ ]:
class AppleEmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            #
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            #
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, embedding_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.projector(x)

        # Put embeddings on the unit sphere.
        # This makes dot product, cosine similarity, and Euclidean distance consistent.
        x = F.normalize(x)
        return x

In [ ]:
class TripletEmbeddingLit(pl.LightningModule):
    def __init__(self, embedding_dim: int = 64, margin: float = 0.2, type_of_triplets: str = "all"):
        super().__init__()
        self.save_hyperparameters()
        self.model = AppleEmbeddingNet(embedding_dim=self.hparams.embedding_dim)
        self.miner = miners.TripletMarginMiner(
            margin=self.hparams.margin, type_of_triplets=self.hparams.type_of_triplets
        )
        self.loss_fn = losses.TripletMarginLoss(margin=self.hparams.margin)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        embeddings = self(x)
        mined_triplets = self.miner(embeddings, y)
        avg_loss = self.loss_fn(embeddings, y, mined_triplets)
        self.log("train_loss", avg_loss, prog_bar=True, on_step=False, on_epoch=True)
        return avg_loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        avg_loss = self.loss_fn(self(x), y)
        self.log("val_loss", avg_loss, prog_bar=True, on_step=False, on_epoch=True)
        return avg_loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters())

### Training

In [ ]:
train_sampler = samplers.MPerClassSampler(
    train_data.targets,
    m=6,
    batch_size=60,
    length_before_new_iter=len(train_data)
)
train_loader = DataLoader(train_data, sampler=train_sampler, batch_size=60)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)
t1_lit_model = TripletEmbeddingLit(embedding_dim=32, margin=0.2)
t1_trainer = pl.Trainer(fast_dev_run=FAST_DEV_RUN, max_epochs=5)
t1_trainer.fit(t1_lit_model, train_loader, test_loader)
task1_model = t1_lit_model.model

### Classification

In [ ]:
def extract_embeddings(dataset, model, batch_size=128):
    device = next(model.parameters()).device

    model.eval()

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_embeddings, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            embeddings = model(images).cpu()
            all_embeddings.append(embeddings)
            all_labels.append(labels)

    return torch.cat(all_embeddings, dim=0), torch.cat(all_labels, dim=0)


def predict_nearest_neighbor(support_embeddings, support_labels, query_embeddings):
    """
    Since embeddings are normalized, the nearest neighbor minimizes Euclidean distance
    and maximizes Cosine Similarity. We can compute this instantly via matrix multiplication.
    query_embeddings: (Q, D), support_embeddings.T: (D, S) -> similarity: (Q, S)
    """
    similarity = torch.mm(query_embeddings, support_embeddings.t())
    closest_indices = similarity.argmax(dim=1)
    return support_labels[closest_indices]

In [ ]:
train_emb, train_lbl = extract_embeddings(train_data, task1_model)
test_emb, test_lbl = extract_embeddings(test_data, task1_model)

seen_preds = predict_nearest_neighbor(train_emb, train_lbl, test_emb)
seen_accuracy = (seen_preds == test_lbl).float().mean().item()

print(f"Seen-item classification accuracy: {seen_accuracy:.4f}")

In [ ]:
support_new_emb, support_new_lbl = extract_embeddings(support_new_data, task1_model)
test_new_emb, test_new_lbl = extract_embeddings(test_new_data, task1_model)

unseen_preds = predict_nearest_neighbor(support_new_emb, support_new_lbl, test_new_emb)
unseen_accuracy = (unseen_preds == test_new_lbl).float().mean().item()

print(f"Unseen-item classification accuracy: {unseen_accuracy:.4f}")

## Task 2  

### Dataset

In [ ]:
class GardenDataset(Dataset):
    def __init__(
        self,
        transform=None,
        class_level="item",
        subset="train",
        family_subset="main",
        item_subset="main",
    ):
        assert class_level in ["item", "family", "both"]
        assert subset in ["train", "test"]
        assert family_subset in ["main", "new", "all"]
        assert item_subset in ["main", "new", "all"]
        base = (
            Path(DATASET_PATH) / "fruits-360_original-size" / "fruits-360-original-size"
        )
        if subset == "train":
            self.path = base / "Training"
        elif subset == "test":
            self.path = base / "Validation"
        self.transform = transform
        self.class_level = class_level

        canonical_items = sorted(
            d
            for d in os.listdir(base / "Training")
            if (base / "Training" / d).is_dir() and re.fullmatch(r"\S+ \d+", d)
        )
        item_to_family = {it: it.rsplit(" ", 1)[0] for it in canonical_items}
        canonical_families = sorted(set(item_to_family.values()))

        self.item_to_idx = {c: i for i, c in enumerate(canonical_items)}
        self.family_to_idx = {c: i for i, c in enumerate(canonical_families)}

        train_fam_to_items = defaultdict(list)
        for it in canonical_items:
            train_fam_to_items[item_to_family[it]].append(it)
        for fam in train_fam_to_items:
            train_fam_to_items[fam].sort(key=lambda x: int(x.rsplit(" ", 1)[1]))

        new_families = {fam for fam, its in train_fam_to_items.items() if len(its) == 1}
        new_items = set()
        for fam, its in train_fam_to_items.items():
            if len(its) >= 3:
                new_items.add(its[0])

        present = {
            d
            for d in os.listdir(self.path)
            if (self.path / d).is_dir() and re.fullmatch(r"\S+ \d+", d)
        }
        all_items = [it for it in canonical_items if it in present]

        if family_subset == "main":
            all_items = [
                it for it in all_items if item_to_family[it] not in new_families
            ]
        elif family_subset == "new":
            all_items = [it for it in all_items if item_to_family[it] in new_families]

        if item_subset == "main":
            all_items = [it for it in all_items if it not in new_items]
        elif item_subset == "new":
            all_items = [it for it in all_items if it in new_items]

        self.items = all_items
        self.item_to_family = {it: item_to_family[it] for it in self.items}
        self.families = sorted(set(self.item_to_family.values()))
        self.new_families = new_families
        self.new_items = new_items

        # Build samples using canonical (global) indices
        self.image_paths = []
        self.targets_item = []
        self.targets_family = []
        for item in self.items:
            item_dir = self.path / item
            item_label = self.item_to_idx[item]
            family_label = self.family_to_idx[item_to_family[item]]
            for img_file in sorted(os.listdir(item_dir)):
                if img_file.endswith(".jpg"):
                    self.image_paths.append(item_dir / img_file)
                    self.targets_item.append(item_label)
                    self.targets_family.append(family_label)

        if class_level == "item":
            self.classes = self.items
            self.class_to_idx = self.item_to_idx
            self.targets = self.targets_item
        elif class_level == "family":
            self.classes = self.families
            self.class_to_idx = self.family_to_idx
            self.targets = self.targets_family

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.class_level == "both":
            return image, self.targets_family[idx], self.targets_item[idx]
        return image, self.targets[idx]

    def get_items_for_family(self, family):
        return sorted(
            item for item, fam in self.item_to_family.items() if fam == family
        )

In [ ]:
%%script echo skipping dataset visualization
dataset = GardenDataset()

family_dd = widgets.Dropdown(options=dataset.families, description="Family:")
item_dd = widgets.Dropdown(
    options=dataset.get_items_for_family(dataset.families[0]), description="Item:"
)
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame:")
output = widgets.Output()


def get_frames(item):
    return sorted(f for f in os.listdir(dataset.path / item) if f.endswith(".jpg"))


def update_items(*_):
    items = dataset.get_items_for_family(family_dd.value)
    item_dd.options = items
    item_dd.value = items[0]


def update_slider(*_):
    frames = get_frames(item_dd.value)
    frame_slider.max = max(0, len(frames) - 1)
    frame_slider.value = min(frame_slider.value, frame_slider.max)
    show_image()


def show_image(*_):
    frames = get_frames(item_dd.value)
    if not frames or frame_slider.value >= len(frames):
        return
    with output:
        clear_output(wait=True)
        img = Image.open(dataset.path / item_dd.value / frames[frame_slider.value])
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(img)
        ax.set_title(
            f"{family_dd.value} | {item_dd.value} | frame {frame_slider.value}"
        )
        ax.axis("off")
        plt.tight_layout()
        plt.show()


family_dd.observe(update_items, names="value")
item_dd.observe(update_slider, names="value")
frame_slider.observe(show_image, names="value")

update_slider()
display(widgets.VBox([family_dd, item_dd, frame_slider, output]))

In [ ]:
train_data = GardenDataset(subset="train", transform=training_transform)

# scenario 1
test_data = GardenDataset(subset="test", transform=training_transform)

# scenario 2
train_data_family = GardenDataset(subset="train", transform=training_transform, class_level="family")
test_data_family = GardenDataset(subset="test", transform=training_transform, class_level="family")

# scenario 3
support_all_data = GardenDataset(subset="train", transform=training_transform, item_subset="all")
test_new_data = GardenDataset(subset="test", transform=training_transform, item_subset="new")

# scenario 4
support_all_data_family = GardenDataset(subset="train", transform=training_transform, family_subset="all", class_level="family")
test_new_data_family = GardenDataset(subset="test", transform=training_transform, family_subset="new", class_level="family")

### Training

In [ ]:
train_sampler = samplers.MPerClassSampler(
    train_data.targets,
    m=6,
    batch_size=60,
    length_before_new_iter=len(train_data)
)

train_loader = DataLoader(train_data, sampler=train_sampler, batch_size=60)
val_loader = DataLoader(test_data, batch_size=128, shuffle=False)

t2_lit_model = TripletEmbeddingLit(embedding_dim=32, margin=0.2, type_of_triplets="all")

t2_trainer = pl.Trainer(fast_dev_run=FAST_DEV_RUN, max_epochs=5)
t2_trainer.fit(t2_lit_model, train_loader, val_loader)

task2_model = t2_lit_model.model

### Classification

In [ ]:
def plot_cm(cm, classes, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    cm_to_plot = cm

    im = ax.imshow(cm_to_plot, cmap="Blues", vmin=0.0, vmax=1.0)
    ax.set_title(title)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45)
    ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted family")
    ax.set_ylabel("True family")

    for i in range(cm_to_plot.shape[0]):
        for j in range(cm_to_plot.shape[1]):
            value = cm_to_plot[i, j] * 100.0
            color = "white" if cm_to_plot[i, j] > 0.5 else "black"
            ax.text(j, i, f"{value:.1f}%", ha="center", va="center", color=color, fontsize=8)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [ ]:
# Scenario 1
train_emb, train_lbl = extract_embeddings(train_data, task2_model)
test_emb, test_lbl = extract_embeddings(test_data, task2_model)

scen1_preds = predict_nearest_neighbor(train_emb, train_lbl, test_emb)
scen1_accuracy = (scen1_preds == test_lbl).float().mean().item()

print(f"Scenario 1 accuracy: {scen1_accuracy:.4f}")

In [ ]:
item_idx_to_family_idx = {
    test_data.class_to_idx[item]: test_data_family.family_to_idx[family]
    for item, family in test_data.item_to_family.items()
}
test_fam_lbl = torch.tensor(
    [item_idx_to_family_idx[label] for label in test_lbl.tolist()]
)
scen1_fam_preds = torch.tensor(
    [item_idx_to_family_idx[label] for label in scen1_preds.tolist()]
)
scenario1_cm = confusion_matrix(
    test_fam_lbl.numpy(), scen1_fam_preds.numpy(), normalize="true"
)
plot_cm(scenario1_cm, test_data_family.classes, "Scenario 1: Family confusion matrix")

In [ ]:
# Scenario 2
train_fam_emb, train_fam_lbl = extract_embeddings(train_data_family, task2_model)
test_fam_emb, test_fam_lbl = extract_embeddings(test_data_family, task2_model)

scen2_preds = predict_nearest_neighbor(train_fam_emb, train_fam_lbl, test_fam_emb)

scenario2_accuracy = (scen2_preds == test_fam_lbl).float().mean().item()
print(f"Scenario 2 accuracy: {scenario2_accuracy:.4f}")

In [ ]:
scenario2_cm = confusion_matrix(
    test_fam_lbl.numpy(), scen2_preds.numpy(), normalize="true"
)
plot_cm(scenario2_cm, test_data_family.classes, "Scenario 2: Family confusion matrix")

In [ ]:
# Scenario 3
supp_all_emb, supp_all_lbl = extract_embeddings(support_all_data, task2_model)
test_new_emb, test_new_lbl = extract_embeddings(test_new_data, task2_model)

scen3_preds = predict_nearest_neighbor(supp_all_emb, supp_all_lbl, test_new_emb)

scenario3_accuracy = (scen3_preds == test_new_lbl).float().mean().item()
print(f"Scenario 3 accuracy: {scenario3_accuracy:.4f}")

In [ ]:
# Scenario 4
supp_all_fam_emb, supp_all_fam_lbl = extract_embeddings(
    support_all_data_family, task2_model
)
test_new_fam_emb, test_new_fam_lbl = extract_embeddings(
    test_new_data_family, task2_model
)

scen4_preds = predict_nearest_neighbor(
    supp_all_fam_emb, supp_all_fam_lbl, test_new_fam_emb
)

scenario4_accuracy = (scen4_preds == test_new_fam_lbl).float().mean().item()
print(f"Scenario 4 accuracy: {scenario4_accuracy:.4f}")

## Task 3

### Dataset

In [ ]:
train_data_both = GardenDataset(
    class_level="both",
    transform=training_transform,
    subset="train",
    family_subset="main",
    item_subset="main",
)

test_data_both = GardenDataset(
    class_level="both",
    transform=training_transform,
    subset="test",
    family_subset="main",
    item_subset="main",
)

### Implementation

In [ ]:
class FamilyAwareTripletLit(pl.LightningModule):
    def __init__(
        self,
        embedding_dim=64,
        margin=0.2,
        family_weight=0.5,
        type_of_triplets="all",
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = AppleEmbeddingNet(embedding_dim=self.hparams.embedding_dim)
        self.item_miner = miners.TripletMarginMiner(
            margin=self.hparams.margin, type_of_triplets=self.hparams.type_of_triplets
        )
        self.family_miner = miners.TripletMarginMiner(
            margin=self.hparams.margin, type_of_triplets=self.hparams.type_of_triplets
        )
        self.loss_fn = losses.TripletMarginLoss(margin=self.hparams.margin)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, family_y, item_y = batch
        embeddings = self(x)

        item_triplets = self.item_miner(embeddings, item_y)
        family_triplets = self.family_miner(embeddings, family_y)

        item_loss = self.loss_fn(embeddings, item_y, item_triplets)
        family_loss = self.loss_fn(embeddings, family_y, family_triplets)
        family_weight = self.hparams.family_weight
        loss = item_loss + family_weight * family_loss

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, family_y, item_y = batch
        embeddings = self(x)

        item_loss = self.loss_fn(embeddings, item_y)
        family_loss = self.loss_fn(embeddings, family_y)
        loss = item_loss + self.hparams.family_weight * family_loss

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters())

### Training

In [ ]:
## for sample i:
# labels[i,0] --> family label
# labels[i,1] --> item label
labels = np.stack(
    [train_data_both.targets_family, train_data_both.targets_item],
     axis=1
)

train_sampler = samplers.HierarchicalSampler(
    labels=labels,
    batch_size=60,
    samples_per_class=6, # number of samples per item
    super_classes_per_batch=5, # number of families per batch
    batches_per_super_tuple=1,
    inner_label= 1, # item labels
    outer_label= 0, # family labels
)

train_loader = DataLoader(train_data_both, batch_sampler=train_sampler)
val_loader = DataLoader(test_data_both, batch_size=128, shuffle=False)

t3_lit_model = FamilyAwareTripletLit(embedding_dim=32, margin=0.2, family_weight=0.5)

t3_trainer = pl.Trainer(fast_dev_run=FAST_DEV_RUN, max_epochs=2)
t3_trainer.fit(t3_lit_model, train_loader, val_loader)

task3_model = t3_lit_model.model

### Classification

In [ ]:
# Scenario 1
train_emb, train_lbl = extract_embeddings(train_data, task3_model)
test_emb, test_lbl = extract_embeddings(test_data, task3_model)

scen1_preds = predict_nearest_neighbor(train_emb, train_lbl, test_emb)

scenario1_accuracy = (scen1_preds == test_lbl).float().mean().item()
print(f"Scenario 1 accuracy: {scenario1_accuracy:.4f}")

In [ ]:
item_idx_to_family_idx = {
    test_data.class_to_idx[item]: test_data_family.family_to_idx[family]
    for item, family in test_data.item_to_family.items()
}
test_fam_lbl = torch.tensor(
    [item_idx_to_family_idx[label] for label in test_lbl.tolist()]
)
scen1_fam_preds = torch.tensor(
    [item_idx_to_family_idx[label] for label in scen1_preds.tolist()]
)
scenario1_cm = confusion_matrix(
    test_fam_lbl.numpy(), scen1_fam_preds.numpy(), normalize="true"
)
plot_cm(scenario1_cm, test_data_family.classes, "Scenario 1: Family confusion matrix")

In [ ]:
# Scenario 2
train_fam_emb, train_fam_lbl = extract_embeddings(train_data_family, task3_model)
test_fam_emb, test_fam_lbl = extract_embeddings(test_data_family, task3_model)

scen2_preds = predict_nearest_neighbor(train_fam_emb, train_fam_lbl, test_fam_emb)

scenario2_accuracy = (scen2_preds == test_fam_lbl).float().mean().item()
print(f"Scenario 2 accuracy: {scenario2_accuracy:.4f}")

scenario2_cm = confusion_matrix(test_fam_lbl.numpy(), scen2_preds.numpy(), normalize="true")
plot_cm(scenario2_cm, test_data_family.classes, "Scenario 2: Family confusion matrix")

In [ ]:
# Scenario 3
supp_all_emb, supp_all_lbl = extract_embeddings(support_all_data, task3_model)
test_new_emb, test_new_lbl = extract_embeddings(test_new_data, task3_model)

scen3_preds = predict_nearest_neighbor(supp_all_emb, supp_all_lbl, test_new_emb)

scenario3_accuracy = (scen3_preds == test_new_lbl).float().mean().item()
print(f"Scenario 3 accuracy: {scenario3_accuracy:.4f}")

In [ ]:
# Scenario 4
supp_all_fam_emb, supp_all_fam_lbl = extract_embeddings(
    support_all_data_family, task3_model
)
test_new_fam_emb, test_new_fam_lbl = extract_embeddings(
    test_new_data_family, task3_model
)

scen4_preds = predict_nearest_neighbor(
    supp_all_fam_emb, supp_all_fam_lbl, test_new_fam_emb
)

scenario4_accuracy = (scen4_preds == test_new_fam_lbl).float().mean().item()
print(f"Scenario 4 accuracy: {scenario4_accuracy:.4f}")

## Task 4

In [ ]:
class BlackoutPixels:
    """Transform that randomly sets x% of pixels to black (0).

    Args:
        fraction: Fraction of pixels to black out (0.0 to 1.0).
    """

    def __init__(self, fraction=0.1):
        self.fraction = fraction

    def __call__(self, img):
        # img shape: (C, H, W)
        _, h, w = img.shape
        num_pixels = h * w
        num_black = int(num_pixels * self.fraction)
        if num_black == 0:
            return img

        # Random pixel indices to black out
        indices = torch.randperm(num_pixels)[:num_black]
        rows = indices // w
        cols = indices % w

        img = img.clone()
        img[:, rows, cols] = 0.0
        return img
    
def get_anomaly_dataset(fraction):
    transform = transforms.Compose([
            training_transform,
            BlackoutPixels(fraction=fraction),
    ])
    return GardenDataset(subset="test", transform=transform, family_subset="main", item_subset="main")
    
    # your code here

In [ ]:
def compute_nn_anomaly_scores(support_embeddings, query_embeddings):
    similarities = torch.mm(query_embeddings, support_embeddings.t())
    return 1.0 - similarities.max(dim=1).values


def get_detection_rate(scores, threshold):
    return (scores > threshold).float().mean().item()

In [ ]:
support_dataset = GardenDataset(
    subset="train",
    transform=training_transform,
    family_subset="main",
    item_subset="main",
)

calibration_dataset = get_anomaly_dataset(0.0)

support_embeddings, _ = extract_embeddings(support_dataset, task3_model)
calibration_embeddings, _ = extract_embeddings(calibration_dataset, task3_model)
normal_scores = compute_nn_anomaly_scores(support_embeddings, calibration_embeddings)
threshold = torch.quantile(normal_scores, 0.9)

print(f"Embedding detector threshold: {threshold.item()}")
print(
    f"False positive rate on clean test set: {get_detection_rate(normal_scores, threshold):.4f}"
)

In [ ]:
for fraction in [0.01, 0.05, 0.10]:
    anomaly_dataset = get_anomaly_dataset(fraction)
    anomaly_embeddings, _ = extract_embeddings(anomaly_dataset, task3_model)
    anomaly_scores = compute_nn_anomaly_scores(support_embeddings, anomaly_embeddings)
    print(
        f"Embedding detector | black pixels {int(fraction * 100)}% | detection rate: {get_detection_rate(anomaly_scores, threshold):.4f}"
    )

In [ ]:
class ConvolutionalAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            #
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            #
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(),
            #
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.ReLU(),
            #
            nn.ConvTranspose2d(16, 3, kernel_size=2, stride=2),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
class AutoencoderLit(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.save_hyperparameters()
        self.model = ConvolutionalAutoencoder()
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, _ = batch
        reconstructions = self(images)
        loss = self.loss_fn(reconstructions, images)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, _ = batch
        reconstructions = self(images)
        loss = self.loss_fn(reconstructions, images)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters())

In [ ]:
calibration_dataset = get_anomaly_dataset(0.0)

ae_train_loader = DataLoader(support_dataset, batch_size=64, shuffle=True)
ae_val_loader = DataLoader(calibration_dataset, batch_size=128, shuffle=False)

ae_lit_model = AutoencoderLit()
ae_trainer = pl.Trainer(fast_dev_run=FAST_DEV_RUN, max_epochs=10)
ae_trainer.fit(ae_lit_model, ae_train_loader, ae_val_loader)

ae_model = ae_lit_model.model

In [ ]:
def reconstruction_scores(model, dataset, batch_size=128):
    device = next(model.parameters()).device
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    scores = []

    model.eval()
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            reconstructions = model(images)
            batch_scores = torch.mean((reconstructions - images) ** 2, dim=(1, 2, 3))
            scores.append(batch_scores.cpu())

    return torch.cat(scores, dim=0)


normal_recon_scores = reconstruction_scores(ae_model, calibration_dataset)
threshold = torch.quantile(normal_recon_scores, 0.9)

print(f"Autoencoder threshold: {threshold.item()}")
print(
    f"False positive rate on clean test set: {get_detection_rate(normal_recon_scores, threshold):.4f}"
)

In [ ]:
for fraction in [0.01, 0.05, 0.10]:
    anomaly_dataset = get_anomaly_dataset(fraction)
    anomaly_scores = reconstruction_scores(ae_model, anomaly_dataset)
    print(
        f"Autoencoder | black pixels {int(fraction * 100)}% | detection rate: {get_detection_rate(anomaly_scores, threshold):.4f}"
    )

In [ ]:
ae_model = ae_lit_model.model

viz_fraction = 0.10  # 10% random black pixels
anomaly_dataset = get_anomaly_dataset(viz_fraction)

idx = 0
img_clean, _ = calibration_dataset[idx]
img_anom, _ = anomaly_dataset[idx]

ae_model.eval()
device = next(ae_model.parameters()).device
with torch.no_grad():
    recon = ae_model(img_clean.unsqueeze(0).to(device)).squeeze(0).cpu()
    recon_anom = ae_model(img_anom.unsqueeze(0).to(device)).squeeze(0).cpu()

mse = torch.mean((recon - img_clean) ** 2).item()

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
axes[0].imshow(img_clean.permute(1, 2, 0).numpy())
axes[0].set_title("Clean")
axes[0].axis("off")

axes[1].imshow(img_anom.permute(1, 2, 0).numpy())
axes[1].set_title(f"Corrupted ({int(viz_fraction * 100)}% black)")
axes[1].axis("off")

axes[2].imshow(recon.clamp(0, 1).permute(1, 2, 0).numpy())
axes[2].set_title(f"Reconstruction clean")
axes[2].axis("off")

axes[3].imshow(recon_anom.clamp(0, 1).permute(1, 2, 0).numpy())
axes[3].set_title(f"Reconstruction corrupted")
axes[3].axis("off")

plt.tight_layout()
plt.show()